<font face="Times New Roman" size=5>
<div dir=rtl align="center">
<font face="Times New Roman" size=5>
<br>
<font color="#008080" size=5>
Analysis of different ML approaches for Multiple Sclerosis (MS) lesion segmentation
<br>
</font>
<font size=4.75>
Melina Rajabi  
<br>
</font>
<font size=3>
Summer 2025



In [1]:

# libraries

import os
import numpy as np
import nibabel as nib
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                            recall_score, f1_score,
                            confusion_matrix, roc_auc_score, RocCurveDisplay)
from skimage.feature import graycomatrix, graycoprops
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from joblib import Parallel, delayed, dump, load

import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from joblib import dump
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from scipy.ndimage import sobel, gaussian_laplace, binary_closing, binary_opening, binary_fill_holes
from skimage.filters import gabor
from skimage.feature import local_binary_pattern
from skimage.morphology import remove_small_objects
from numpy.lib.stride_tricks import sliding_window_view

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.kernel_approximation import RBFSampler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import fbeta_score

import warnings
warnings.filterwarnings('ignore')

In [2]:

# Base code

def data_cleaning(data):

    if data is None:
        return data
    
    data = np.asarray(data)
    
    if data.size == 0:
        return data.astype(np.float32, copy=False)
    
    finite = np.isfinite(data)
    
    if not finite.any():
        return np.zeros_like(data, dtype=np.float32)
    
    finite_vals = data[finite]
    nan_fill = float(np.median(finite_vals)) if finite_vals.size else 0.0
    pos_fill = float(np.percentile(finite_vals, 99.9)) if finite_vals.size else 0.0
    neg_fill = float(np.percentile(finite_vals, 0.1)) if finite_vals.size else 0.0
    data = np.nan_to_num(data, nan=nan_fill, posinf=pos_fill, neginf=neg_fill)
    
    finite = np.isfinite(data)
    
    if not finite.any():
        return np.zeros_like(data, dtype=np.float32)
    
    finite_vals = data[finite]
    upper = float(np.percentile(finite_vals, 99.5)) if finite_vals.size else 0.0
    lower = float(np.percentile(finite_vals, 0.5)) if finite_vals.size else 0.0

    return np.clip(data, lower, upper).astype(np.float32, copy=False)


def load_train_data(file_path):

    print('Loading train data...')
    patients = sorted(os.listdir(file_path))
    X_train = []
    y_train = []
    for p in (patients):
        p_path = os.path.join(file_path, p)

        for t in tqdm (os.listdir(p_path)):
            t_path = os.path.join(p_path, t)

            flair_file = f"{p}_{t}_FLAIR.nii.gz"
            mask_file  = f"{p}_{t}_MASK.nii.gz"
            t1_file    = f"{p}_{t}_T1.nii.gz"
            t2_file    = f"{p}_{t}_T2.nii.gz"

            flair_path = os.path.join(t_path, flair_file)
            mask_path  = os.path.join(t_path, mask_file)
            t1_path    = os.path.join(t_path, t1_file)
            t2_path    = os.path.join(t_path, t2_file)


            flair = nib.load(flair_path).get_fdata()
            mask = nib.load(mask_path).get_fdata()
            t1 = nib.load(t1_path).get_fdata()
            t2 = nib.load(t2_path).get_fdata()

            flair, t1, t2 = normalize(data_cleaning(flair), data_cleaning(t1), data_cleaning(t2))
            mask = mask.astype(np.uint8)

            for slice in range (flair.shape[2]):

                if np.sum(mask[:, :, slice]) > 5:
                    feature = extract_features(flair[:, :, slice], t1[:, :, slice], t2[:, :, slice])
                    label = (mask[:, : , slice]).flatten()

                    pos_idx = np.where(label == 1)[0]
                    neg_idx = np.where(label == 0)[0]

                    if len(pos_idx) > 0:

                        n_samples = min(len(pos_idx)*3, len(neg_idx))
                        sel_neg = np.random.choice(neg_idx, size=n_samples, replace=False)
                        sel_idx = np.concatenate([pos_idx, sel_neg])

                        X_train.append(feature[sel_idx])
                        y_train.append(label[sel_idx])


    X_train = np.concatenate(X_train, axis=0)
    y_train = np.concatenate(y_train, axis=0)

    return X_train, y_train


def load_test_data(file_path):

    print('Loading test data...')
    patients = sorted(os.listdir(file_path))
    test_files = []

    for p in tqdm(patients):
        p_path = os.path.join(file_path, p)

        flair = nib.load(os.path.join(p_path, f"{p}_FLAIR.nii.gz")).get_fdata()
        mask  = nib.load(os.path.join(p_path, f"{p}_MASK.nii.gz")).get_fdata()
        t1    = nib.load(os.path.join(p_path, f"{p}_T1.nii.gz")).get_fdata()
        t2    = nib.load(os.path.join(p_path, f"{p}_T2.nii.gz")).get_fdata()

        flair, t1, t2 = normalize(flair, t1, t2)

        test_files.append({
            'patient': p,
            'flair': flair,
            'mask': mask,
            't1': t1,
            't2': t2
        })

    return test_files




def extract_patch(image, x, y, size=5):

    half = size // 2
    h, w = image.shape
    patch = np.zeros((size, size))
    xmin, xmax = max(0, x-half), min(w, x+half+1)
    ymin, ymax = max(0, y-half), min(h, y+half+1)
    patch[(ymin-y+half):(ymax-y+half), (xmin-x+half):(xmax-x+half)] = image[ymin:ymax, xmin:xmax]

    return patch


def zscore_scale(image):

    image =data_cleaning(image)
    mean = np.mean(image)
    std = max(np.std(image), 1e-6)
    return (image - mean) / std


def robust_scale(image):

    image = data_cleaning(image)
    median = np.median(image)
    q75, q25 = np.percentile(image, [75, 25])
    iqr = max(q75 - q25, 1e-6)

    return (image - median) / iqr


def normalize(flair, t1, t2):

    normalized = []
    normalized.append(robust_scale(flair))  # FLAIR
    normalized.append(zscore_scale(t1))  # T1
    normalized.append(zscore_scale(t2))  # T2

    return normalized



def extract_features(flair_slice, t1_slice, t2_slice, patch_size=5, small_patch_size=3):
  
    # Ensure float32 for speed
    flair = flair_slice.astype(np.float32, copy=False)
    t1 = t1_slice.astype(np.float32, copy=False)
    t2 = t2_slice.astype(np.float32, copy=False)

    h, w = flair.shape
    npix = h * w

    # Basic gradients and LoG 
    grad_f = np.abs(sobel(flair))
    grad_t1 = np.abs(sobel(t1))
    grad_t2 = np.abs(sobel(t2))

    log_f = np.abs(gaussian_laplace(flair, sigma=1))
    log_t1 = np.abs(gaussian_laplace(t1, sigma=1))
    log_t2 = np.abs(gaussian_laplace(t2, sigma=1))

    # Gabor features 
    gabor_feats = []
    for freq in [0.2, 0.4]:
        for theta in [0, np.pi/4]:
            try:
                _, tex = gabor(flair, frequency=freq, theta=theta)
                gabor_feats.append(tex.astype(np.float32))
            except Exception:
                gabor_feats.append(np.zeros_like(flair, dtype=np.float32))

    # LBP multi-radius
    lbp_feats = []
    for R in [1, 2]:
        lbp_feats.append(local_binary_pattern(flair, P=8, R=R, method='uniform').astype(np.float32))

    # Intensity ratios
    ratio1 = flair / (t1 + 1e-8)
    ratio2 = flair / (t2 + 1e-8)

    # Coordinates in [-1,1]
    yy, xx = np.meshgrid(np.linspace(-1, 1, h), np.linspace(-1, 1, w), indexing='ij')

   
    pad = patch_size // 2
    pad_small = small_patch_size // 2

    def pad_and_sw(img, p):
        if p == 1:
            # no patch
            return img.reshape(h, w, 1, 1)
        padded = np.pad(img, pad_width=p//2, mode='reflect')
        windows = sliding_window_view(padded, (p, p))
        # windows shape: (h, w, p, p)
        return windows.reshape(h, w, p, p)

    # Large patches
    p_flair = pad_and_sw(flair, patch_size) 
    p_t1 = pad_and_sw(t1, patch_size)
    p_t2 = pad_and_sw(t2, patch_size)

    # Small patches for gradients
    sg_flair = pad_and_sw(flair, small_patch_size)
    sg_grad_f = pad_and_sw(grad_f, small_patch_size)

    # Compute patch statistics (mean, std, median, min, max, range) for each pixel
    def patch_stats(patch_array):
       
        arr = patch_array.reshape(h, w, -1)
        mean = arr.mean(axis=2)
        std = arr.std(axis=2)
        med = np.median(arr, axis=2)
        mn = arr.min(axis=2)
        mx = arr.max(axis=2)
        rng = mx - mn
        # center - mean
        center_idx = arr.shape[2] // 2
        center = arr[:, :, center_idx]
        ctr_minus_mean = center - mean
        return np.stack([mean, std, med, mn, mx, rng, center, ctr_minus_mean], axis=-1)  # shape (h,w,8)

    stats_flair = patch_stats(p_flair)
    stats_t1 = patch_stats(p_t1)
    stats_t2 = patch_stats(p_t2)

    # small patch flattened descriptor but keep it short: central 3x3 flattened
    spf = sg_flair.reshape(h, w, -1)
    if spf.shape[2] > 9:
        spf = spf[:, :, :9]
    spg = sg_grad_f.reshape(h, w, -1)
    if spg.shape[2] > 9:
        spg = spg[:, :, :9]

    # For gradients we compute central pixel + mean patch gradient + std
    grad_stats_f = np.stack([
        grad_f,
        grad_f.mean() * np.ones_like(grad_f)  # global mean (cheap placeholder) - keep consistent sizing
    ], axis=-1)

    # Prepare per-pixel features consolidated:
    # We'll build them in order, then reshape to (h*w, F)
    feat_list = []

    # central intensities
    feat_list.append(flair.reshape(h, w, 1))
    feat_list.append(t1.reshape(h, w, 1))
    feat_list.append(t2.reshape(h, w, 1))

    # patch statistics for each modality (8 each)
    feat_list.append(stats_flair)  # 8
    feat_list.append(stats_t1)     # 8
    feat_list.append(stats_t2)     # 8

    # gradient magnitudes at central pixel and LoG
    feat_list.append(grad_f.reshape(h, w, 1))
    feat_list.append(grad_t1.reshape(h, w, 1))
    feat_list.append(grad_t2.reshape(h, w, 1))
    feat_list.append(log_f.reshape(h, w, 1))
    feat_list.append(log_t1.reshape(h, w, 1))
    feat_list.append(log_t2.reshape(h, w, 1))

    # small-patch flattened features (up to 9 each)
    feat_list.append(spf)  # small raw neighborhood
    feat_list.append(spg)  # small gradient neighborhood

    # gabor responses (one value per orientation/freq)
    for g in gabor_feats:
        feat_list.append(g.reshape(h, w, 1))

    # LBP features
    for lb in lbp_feats:
        feat_list.append(lb.reshape(h, w, 1))

    # Ratios
    feat_list.append(ratio1.reshape(h, w, 1))
    feat_list.append(ratio2.reshape(h, w, 1))

    # Coordinates last (preserve the ordering from your original code - last two are coords)
    feat_list.append(xx.reshape(h, w, 1))
    feat_list.append(yy.reshape(h, w, 1))

    # Concatenate along feature axis
    feats = np.concatenate(feat_list, axis=2)  # shape (h, w, F)

    # Flatten to (h*w, F)
    F = feats.shape[2]
    feats_flat = feats.reshape(npix, F).astype(np.float32, copy=False)

    # final cleaning and clipping
    feats_flat = data_cleaning(feats_flat)

    return feats_flat


def post_process(pred_mask, min_size=25):

    pred_mask = pred_mask.astype(bool)

    pred_mask = binary_opening(pred_mask, structure=np.ones((3,3,3)))
    pred_mask = binary_closing(pred_mask, structure=np.ones((5,5,5)))
    pred_mask = binary_fill_holes(pred_mask)


    pred_mask = remove_small_objects(pred_mask, min_size=min_size)

    return pred_mask.astype(np.uint8)


def test_model_predict(model, test_files, scaler, model_type):
    print('Testing model ...')
    results = []

    for file in tqdm(test_files):
        flair, t1, t2, mask = file['flair'], file['t1'], file['t2'], file['mask']
        pred_probs = np.zeros_like(mask, dtype=np.float32)

        for i in range(flair.shape[2]):
            features = extract_features(flair[:, :, i], t1[:, :, i], t2[:, :, i])
            features = scaler.transform(features)

            if model_type == 'xgboost':
                dtest = xgb.DMatrix(features)
                preds = model.predict(dtest)

            elif model_type == 'svm':
                preds = model.decision_function(features) 
                preds = 1 / (1 + np.exp(-preds))  

            elif model_type in ['adaboost', 'logistic', 'rbf']:
                preds = model.predict_proba(features)[:, 1]  
            else:
                raise ValueError(f"Unknown model type: {model_type}")

            pred_probs[:, :, i] = preds.reshape(flair.shape[:2])

        # Threshold optimization
        thresholds = np.linspace(0.1, 0.5, 10)
        best_thresh = 0.3
        best_f1 = 0
        for thresh in thresholds:
            temp_pred = (pred_probs > thresh).astype(int)
            current_f1 = f1_score(mask.flatten(), temp_pred.flatten())
            
            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = thresh

        pred_test = (pred_probs > best_thresh).astype(int)
        pred_test = post_process(pred_test)

        accuracy = accuracy_score(mask.flatten(), pred_test.flatten())
        f1 = f1_score(mask.flatten(), pred_test.flatten())
        dice = 2 * np.sum(mask * pred_test) / (np.sum(mask) + np.sum(pred_test) + 1e-8)

        results.append({
            'patient': file['patient'],
            'pred_mask': pred_test,
            'true_mask': mask,
            'flair': flair,
            'accuracy': accuracy,
            'f1_score': f1,
            'dice_score': dice,
            'optimal_threshold': best_thresh
        })

        visualize_results(results[-1])

    return results


def visualize_results(result):
    slice_idx = result['flair'].shape[2] // 2
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 4, 1)
    plt.imshow(result['flair'][:, :, slice_idx], cmap='gray')
    plt.title('FLAIR Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(result['flair'][:, :, slice_idx], cmap='gray')
    plt.imshow(result['pred_mask'][:, :, slice_idx], cmap='jet', alpha=0.4)
    plt.title(f'Prediction (F1: {result["f1_score"]:.2f})')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(result['flair'][:, :, slice_idx], cmap='gray')
    plt.imshow(result['true_mask'][:, :, slice_idx], cmap='jet', alpha=0.4)
    plt.title('Ground Truth')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(np.abs(result['pred_mask'][:, :, slice_idx] - result['true_mask'][:, :, slice_idx]),
               cmap='bwr', vmin=0, vmax=1)
    plt.title('Error Map')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


SVM Model

In [4]:

# SVM

def SVM_train(X_train, y_train):
    print('Training the SVM model...')
    X_raw = X_train
    is_brain = X_raw[:, -1] > 0.5  

    pos_mask = (y_train == 1)
    neg_mask = ~pos_mask
    bg_neg_mask = neg_mask & (~is_brain)
    brain_neg_mask = neg_mask & is_brain

    transformer = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95, svd_solver='auto', random_state=42))
    ])
    X_trans = transformer.fit_transform(X_raw)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_trans, y_train, test_size=0.15, stratify=y_train, random_state=42
    )

    Cs = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
    pos_multipliers = [3.0, 4.0, 6.0]
    bg_multipliers = [3.0, 5.0, 8.0]

    best_f1, best_cfg = -1.0, None
    for pos_mul in pos_multipliers:
        for bg_mul in bg_multipliers:
            sw_full = np.ones_like(y_train, dtype=float)
            sw_full[pos_mask] *= pos_mul
            sw_full[bg_neg_mask] *= bg_mul
            sw_full[brain_neg_mask] *= 1.0
            sw_full *= (len(sw_full) / sw_full.sum())

      
            rng = np.random.RandomState(42)
            perm = rng.permutation(len(y_train))
            tr_sel = np.zeros(len(y_train), dtype=bool)
            tr_sel[perm[:len(y_tr)]] = True
            sw_tr = sw_full[tr_sel]

            for C in Cs:
                clf = LinearSVC(
                    C=C,
                    loss='squared_hinge', 
                    class_weight=None,
                    random_state=42,
                    max_iter=30000,
                    dual=False,
                    tol=5e-5,
                    verbose=0
                )
                clf.fit(X_tr, y_tr, sample_weight=sw_tr)
                y_v_pred = (clf.decision_function(X_val) > 0).astype(int)
                f1 = f1_score(y_val, y_v_pred)
                if f1 > best_f1:
                    best_f1 = f1
                    best_cfg = (C, pos_mul, bg_mul)

    best_C, best_pos_mul, best_bg_mul = best_cfg
    sample_weight = np.ones_like(y_train, dtype=float)
    sample_weight[pos_mask] *= best_pos_mul
    sample_weight[bg_neg_mask] *= best_bg_mul
    sample_weight[brain_neg_mask] *= 1.0
    sample_weight *= (len(sample_weight) / sample_weight.sum())

    model = LinearSVC(
        C=best_C,
        loss='squared_hinge',
        class_weight=None,
        random_state=42,
        max_iter=30000,
        dual=False,
        tol=5e-5,
        verbose=1
    )
    model.fit(X_trans, y_train, sample_weight=sample_weight)

    dump(model, 'svm_model.joblib')
    dump(transformer, 'scaler.joblib')
    return model, transformer


XGBoost Model

In [5]:

# XGBOOST

def XGBOOST_train(X_train, y_train):

    print('Training the XGBOOST model...')
    X_train = data_cleaning(X_train)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    dtrain = xgb.DMatrix(X_train, label=y_train)

    params = {
        'objective': 'binary:logistic',
        'eval_metric': ['logloss', 'auc'],
        'eta': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42,
        'scale_pos_weight': np.sum(y_train == 0) / np.sum(y_train == 1),
        'lambda': 1.0,
        'alpha': 0.5,
        'tree_method': 'hist' 
    }

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=500,
        verbose_eval=10
    )


    dump(model, 'xgboost_model.joblib')
    dump(scaler, 'scaler.joblib')

    return model, scaler


AdaBoost Model

In [6]:

# adaboost

def _stratified_subsample(X, y, max_samples=250_000, random_state=42):
    if X.shape[0] <= max_samples:
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=max_samples, random_state=random_state)
    idx, _ = next(sss.split(X, y))
    return X[idx], y[idx]

def adaboost_train(X_train, y_train,
                   max_fit_samples=250_000,
                   val_size=0.15,
                   max_estimators=300,
                   learning_rate=0.2,
                   random_state=42):

    print("\nTraining AdaBoost model (fast, staged early-stopping)...")

    X_train = data_cleaning(X_train)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)

    X_sub, y_sub = _stratified_subsample(X_train, y_train, max_samples=max_fit_samples, random_state=random_state)

  
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    train_idx, val_idx = next(sss.split(X_sub, y_sub))
    X_tr, y_tr = X_sub[train_idx], y_sub[train_idx]
    X_val, y_val = X_sub[val_idx], y_sub[val_idx]

    sw_tr = compute_sample_weight('balanced', y_tr)

    base_tree = DecisionTreeClassifier(
        criterion='gini',
        max_depth=2,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=random_state
    )

   
    model = AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=max_estimators,
        learning_rate=learning_rate,
        algorithm='SAMME.R',
        random_state=random_state
    )
    model.fit(X_tr, y_tr, sample_weight=sw_tr)

   
    best_k, best_f1 = 1, -1.0
    for k, y_pred in enumerate(model.staged_predict(X_val), start=1):
        f1 = f1_score(y_val, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_k = k

    print(f"Best stage k={best_k}/{max_estimators}, val F1={best_f1:.4f}")

    sw_all = compute_sample_weight('balanced', y_sub)
    final_model = AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=best_k,
        learning_rate=learning_rate,
        algorithm='SAMME.R',
        random_state=random_state
    )
    final_model.fit(X_sub, y_sub, sample_weight=sw_all)

    dump(final_model, 'adaboost_model.joblib')
    dump(scaler, 'scaler.joblib')
    return final_model, scaler


RBF Model

In [7]:

# RBF

def RBF_train(X_train, y_train, sample_size=10000):

    print(f"\nTraining fast RBF SVM model with {sample_size} samples...")


    if len(X_train) > sample_size:
        print(f"Sampling {sample_size} samples from {len(X_train)} total samples...")

        
        from sklearn.model_selection import train_test_split
        X_sample, _, y_sample, _ = train_test_split(
            X_train, y_train,
            train_size=sample_size,
            stratify=y_train,
            random_state=42
        )
    else:
        X_sample, y_sample = X_train, y_train


    scaler = StandardScaler()
    X_sample_scaled = scaler.fit_transform(X_sample)


    model = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=42,
        cache_size=2000,
        verbose=1
   )

  
    model.fit(X_sample_scaled, y_sample)

    dump(model, 'rbf_svm_model_fast.joblib')
    dump(scaler, 'rbf_scaler_fast.joblib')

    print("Fast RBF SVM training completed!")
    return model, scaler


Logistic Regression Model

In [8]:

# logistic regression

def _stratified_subsample_with_idx(X, y, max_samples=250_000, random_state=42):
    if X.shape[0] <= max_samples:
        idx = np.arange(X.shape[0])
        return X, y, idx
    
    sss = StratifiedShuffleSplit(n_splits=1, train_size=max_samples, random_state=random_state)
    idx, _ = next(sss.split(X, y))
    return X[idx], y[idx], idx

def _median_heuristic_gamma(X, max_samples=20000, random_state=42):
    if X.shape[0] > max_samples:
        Xs, _, _ = _stratified_subsample_with_idx(X, np.zeros(X.shape[0], dtype=int), max_samples, random_state)
    else:
        Xs = X
    var = float(np.var(Xs))
    var = max(var, 1e-12)
    return 1.0 / (Xs.shape[1] * var)

def _center_weights_from_coords(coords, alpha=2.0):
    
    r2 = (coords[:, 0] ** 2 + coords[:, 1] ** 2)
    return np.exp(-alpha * r2)

def _small_lesion_weights_from_coords(y, coords, bins=48, power=0.5, boost=3.0):
  
    x = np.clip(((coords[:, 0] + 1.0) * 0.5 * (bins - 1)).astype(int), 0, bins - 1)
    yb = np.clip(((coords[:, 1] + 1.0) * 0.5 * (bins - 1)).astype(int), 0, bins - 1)
    flat_idx = x * bins + yb

    pos_mask = (y == 1)
    counts = np.zeros(bins * bins, dtype=np.int32)
    np.add.at(counts, flat_idx[pos_mask], 1)

    
    pos_counts = counts[flat_idx].astype(np.float32)
    w = np.ones_like(pos_counts, dtype=np.float32)
    w[pos_mask] = boost / (np.power(np.maximum(pos_counts[pos_mask], 1.0), power))
    return w

def LogisticRegression_train(X_train, y_train,
                             max_search_samples=120_000,
                             max_fit_samples=400_000,
                             val_size=0.2,
                             center_alpha=2.0,
                             center_boost=1.5,
                             small_boost=3.0,
                             bins=48,
                             random_state=42):

    print("\nTraining logistic regression (RBF, small-lesion weighted, F2-tuned)...")

    
    X_raw = data_cleaning(X_train)
    coords_all = X_raw[:, -2:] 
    cw_all = _center_weights_from_coords(coords_all, alpha=center_alpha)
    slw_all = _small_lesion_weights_from_coords(y_train, coords_all, bins=bins, power=0.5, boost=small_boost)

  
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw).astype(np.float32, copy=False)

 
    X_search, y_search, idx_search = _stratified_subsample_with_idx(X_scaled, y_train, max_search_samples, random_state)
    X_fitcap, y_fitcap, idx_fitcap = _stratified_subsample_with_idx(X_scaled, y_train, max_fit_samples, random_state)

    
    cw_search, slw_search = cw_all[idx_search], slw_all[idx_search]
    cw_fitcap, slw_fitcap = cw_all[idx_fitcap], slw_all[idx_fitcap]

    
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    tr_idx, val_idx = next(sss.split(X_search, y_search))
    X_tr, y_tr = X_search[tr_idx], y_search[tr_idx]
    X_val, y_val = X_search[val_idx], y_search[val_idx]
    cw_tr, cw_val = cw_search[tr_idx], cw_search[val_idx]
    slw_tr, slw_val = slw_search[tr_idx], slw_search[val_idx]


    base_sw = compute_sample_weight('balanced', y_tr).astype(np.float32)
    sw_tr = base_sw * (1.0 + center_boost * cw_tr) * slw_tr

    gamma0 = _median_heuristic_gamma(X_tr)

    n_components_list = [1000, 1500, 2000]
    Cs = [0.3, 1.0, 3.0, 10.0]
    thresholds = np.linspace(0.1, 0.9, 33)
    beta = 2.0  

    best = dict(f1=-1.0)
    for n_comp in n_components_list:
        for C in Cs:
            model = make_pipeline(
                RBFSampler(gamma=gamma0, n_components=n_comp, random_state=42),
                LogisticRegression(
                    penalty='l2',
                    C=C,
                    solver='lbfgs',
                    max_iter=1200,
                    tol=1e-3,
                    n_jobs=-1,
                    random_state=random_state
                )
            )
            model.fit(X_tr, y_tr, logisticregression__sample_weight=sw_tr)

            p = model.predict_proba(X_val)[:, 1]
            val_w = compute_sample_weight('balanced', y_val).astype(np.float32)
            val_w = val_w * (1.0 + center_boost * cw_val) * slw_val

            scores = [fbeta_score(y_val, (p > t).astype(int), beta=beta, sample_weight=val_w, zero_division=0)
                      for t in thresholds]
            i = int(np.argmax(scores))
            if scores[i] > best['f1']:
                best.update(model=model, n_components=n_comp, C=C,
                            thresh=float(thresholds[i]), f1=float(scores[i]))

    print(f"Best LR-RBF (small-lesion): n_comp={best['n_components']}, C={best['C']}, "
          f"val F2_wt={best['f1']:.4f}, thr={best['thresh']:.2f}")

    
    sw_all = compute_sample_weight('balanced', y_fitcap).astype(np.float32)
    sw_all = sw_all * (1.0 + center_boost * cw_fitcap) * slw_fitcap

    final_model = make_pipeline(
        RBFSampler(gamma=gamma0, n_components=best['n_components'], random_state=42),
        LogisticRegression(
            penalty='l2',
            C=best['C'],
            solver='lbfgs',
            max_iter=1500,
            tol=1e-3,
            n_jobs=-1,
            random_state=random_state
        )
    )
    final_model.fit(X_fitcap, y_fitcap, logisticregression__sample_weight=sw_all)

    dump(final_model, 'logreg_model.joblib')
    dump(scaler, 'scaler.joblib')
    return final_model, scaler



Load train data

In [9]:

train_path = "C:\\Users\\melin\\Desktop\\project - 2025\\main\\data\\MSLesSeg Dataset\\MSLesSeg Dataset\\train"
test_path = "C:\\Users\\melin\\Desktop\\project - 2025\\main\\data\\MSLesSeg Dataset\\MSLesSeg Dataset\\test"
    
# Load data
X_train, y_train = load_train_data(train_path)

Loading train data...


100%|██████████| 3/3 [00:49<00:00, 16.52s/it]



Load test data

In [10]:

test_files = load_test_data(test_path)

Loading test data...


100%|██████████| 22/22 [01:21<00:00,  3.69s/it]



SVM Result

In [20]:

# Train model
model, scaler = SVM_train(X_train, y_train)

Training the SVM model...
[LibLinear]

In [ ]:

# Test model
results = test_model_predict(model, test_files, scaler,'svm')
    
# Print summary statistics
print('SVM MODEL:\n')
avg_f1 = np.mean([r['f1_score'] for r in results])
avg_dice = np.mean([r['dice_score'] for r in results])
print(f"\nAverage F1 Score: {avg_f1:.3f}")
print(f"Average Dice Score: {avg_dice:.3f}")


AdaBoost Result

In [16]:

ada_model, scaler = adaboost_train(X_train, y_train)


Training AdaBoost model (fast, staged early-stopping)...
Best stage k=296/300, val F1=0.8332


In [ ]:

results = test_model_predict(ada_model, test_files, scaler, 'adaboost')
# Print summary statistics
print('ADABOOST MODEL:\n')
avg_f1 = np.mean([r['f1_score'] for r in results])
avg_dice = np.mean([r['dice_score'] for r in results])
print(f"\nAverage F1 Score: {avg_f1:.3f}")
print(f"Average Dice Score: {avg_dice:.3f}")


Logistic Regression Result

In [14]:

log_model, scaler = LogisticRegression_train(X_train, y_train)


Training logistic regression (RBF, small-lesion weighted, F2-tuned)...
Best LR-RBF (small-lesion): n_comp=1000, C=3.0, val F2_wt=0.5977, thr=0.15


In [ ]:

# Test model
results = test_model_predict(log_model, test_files, scaler, 'logistic')
    
# Print summary statistics
print('LOGISTIC REGRESSION MODEL:\n')
avg_f1 = np.mean([r['f1_score'] for r in results])
avg_dice = np.mean([r['dice_score'] for r in results])
print(f"\nAverage F1 Score: {avg_f1:.3f}")
print(f"Average Dice Score: {avg_dice:.3f}")


RBF Result

In [18]:

rbf_model, scaler = RBF_train(X_train, y_train)


Training fast RBF SVM model with 10000 samples...
Sampling 10000 samples from 4523304 total samples...
[LibSVM]Fast RBF SVM training completed!


In [ ]:

# Test model
results = test_model_predict(rbf_model, test_files, scaler, 'rbf')
    
# Print summary statistics
print('RBF MODEL:\n')
avg_f1 = np.mean([r['f1_score'] for r in results])
avg_dice = np.mean([r['dice_score'] for r in results])
print(f"\nAverage F1 Score: {avg_f1:.3f}")
print(f"Average Dice Score: {avg_dice:.3f}")


XGBoost Result

In [12]:

# Train model
model, scaler = XGBOOST_train(X_train, y_train)

Training the XGBOOST model...


In [ ]:

# Test model
results = test_model_predict(model, test_files, scaler, 'xgboost')
    
# Print summary statistics
print('XGBOOST MODEL:\n')
avg_f1 = np.mean([r['f1_score'] for r in results])
avg_dice = np.mean([r['dice_score'] for r in results])
print(f"\nAverage F1 Score: {avg_f1:.3f}")
print(f"Average Dice Score: {avg_dice:.3f}")